# WiDS 2026 — Extreme Hybrid Survival Stack

In [1]:

# ============================================
# WiDS 2026 — Extreme Hybrid Survival Stack
# ============================================

import os
import gc
import math
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.special import expit, logit
from scipy.stats import norm
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

try:
    from catboost import CatBoostClassifier, Pool
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False

try:
    import xgboost as xgb
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


SEED = 2026
HORIZONS = [12, 24, 48, 72]
N_SPLITS = 5
N_REPEATS = 4
EPS = 1e-7
N_SEARCH = 5000
N_NEARFAR_GRID = 7


def seed_everything(seed: int = 2026) -> None:
    np.random.seed(seed)


seed_everything(SEED)


def find_dataset_root(base: str = "/kaggle/input") -> Path:
    base_path = Path(base)
    if not base_path.exists():
        raise FileNotFoundError("'/kaggle/input' not found. Attach the competition data or the GitHub repo as an input dataset.")
    required = ["train.csv", "test.csv", "sample_submission.csv", "metaData.csv"]
    candidate_dirs = set()
    for p in base_path.rglob("train.csv"):
        candidate_dirs.add(p.parent)
        for anc in p.parents:
            if str(anc).startswith(str(base_path)):
                candidate_dirs.add(anc)
    if not candidate_dirs:
        raise FileNotFoundError("Could not locate train.csv under /kaggle/input.")
    scored = []
    for d in candidate_dirs:
        count = sum((d / f).exists() for f in required)
        scored.append((count, len(d.parts), -len(str(d)), d))
    scored.sort(reverse=True)
    root = scored[0][-1]
    missing = [f for f in required if not (root / f).exists()]
    if missing:
        # fallback: grab files individually by shortest path
        found = {}
        for fname in required:
            matches = list(base_path.rglob(fname))
            if not matches:
                raise FileNotFoundError(f"Could not locate {fname} under /kaggle/input.")
            matches = sorted(matches, key=lambda p: (len(p.parts), len(str(p))))
            found[fname] = matches[0]
        return found["train.csv"].parent
    return root


def locate_file(root: Path, fname: str) -> Path:
    if (root / fname).exists():
        return root / fname
    matches = sorted(Path("/kaggle/input").rglob(fname), key=lambda p: (len(p.parts), len(str(p))))
    if not matches:
        raise FileNotFoundError(f"Could not locate {fname} under /kaggle/input.")
    return matches[0]

def read_competition_files():
    root = find_dataset_root()
    train = pd.read_csv(locate_file(root, "train.csv"))
    test = pd.read_csv(locate_file(root, "test.csv"))
    sample = pd.read_csv(locate_file(root, "sample_submission.csv"))
    meta = pd.read_csv(locate_file(root, "metaData.csv"))
    for df in [train, test, sample]:
        if "event_id" in df.columns:
            df["event_id"] = df["event_id"].astype(str)
    print(f"Dataset root: {root}")
    print(f"Train shape: {train.shape} | Test shape: {test.shape}")
    return root, train, test, sample, meta


def make_strata(event: np.ndarray, time: np.ndarray) -> np.ndarray:
    strata = np.empty(len(event), dtype=object)
    for cls in [0, 1]:
        mask = event == cls
        vals = pd.Series(time[mask])
        if mask.sum() == 0:
            continue
        n_unique = vals.nunique(dropna=True)
        q = min(4, int(n_unique)) if n_unique > 1 else 1
        if q > 1:
            bins = pd.qcut(vals, q=q, labels=False, duplicates="drop").astype(int).values
        else:
            bins = np.zeros(mask.sum(), dtype=int)
        strata[mask] = [f"{cls}_{b}" for b in bins]
    return strata.astype(str)


def known_and_label(time: np.ndarray, event: np.ndarray, horizon: int):
    known = (event == 1) | (time >= horizon)
    label = ((event == 1) & (time <= horizon)).astype(int)
    return known, label


def brier_at_horizon(time: np.ndarray, event: np.ndarray, pred: np.ndarray, horizon: int) -> float:
    known, label = known_and_label(time, event, horizon)
    if known.sum() == 0:
        return 0.25
    return float(np.mean((pred[known] - label[known]) ** 2))


def harrell_c_index(time: np.ndarray, event: np.ndarray, risk: np.ndarray) -> float:
    n = len(time)
    permissible = 0.0
    concordant = 0.0
    ties = 0.0
    for i in range(n):
        ti, ei, ri = time[i], event[i], risk[i]
        for j in range(i + 1, n):
            tj, ej, rj = time[j], event[j], risk[j]
            if ei == 1 and ej == 1:
                if ti == tj:
                    continue
                permissible += 1
                if ti < tj:
                    if ri > rj:
                        concordant += 1
                    elif ri == rj:
                        ties += 1
                else:
                    if rj > ri:
                        concordant += 1
                    elif ri == rj:
                        ties += 1
            elif ei == 1 and tj > ti:
                permissible += 1
                if ri > rj:
                    concordant += 1
                elif ri == rj:
                    ties += 1
            elif ej == 1 and ti > tj:
                permissible += 1
                if rj > ri:
                    concordant += 1
                elif ri == rj:
                    ties += 1
    if permissible == 0:
        return 0.5
    return float((concordant + 0.5 * ties) / permissible)


def monotone_clip(pred: np.ndarray) -> np.ndarray:
    pred = np.clip(pred, 0.0, 1.0)
    pred = np.maximum.accumulate(pred, axis=1)
    pred = np.clip(pred, 0.0, 1.0)
    return pred


def hybrid_score(time: np.ndarray, event: np.ndarray, pred: np.ndarray):
    pred = monotone_clip(np.asarray(pred, dtype=float))
    b24 = brier_at_horizon(time, event, pred[:, 1], 24)
    b48 = brier_at_horizon(time, event, pred[:, 2], 48)
    b72 = brier_at_horizon(time, event, pred[:, 3], 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    cidx = harrell_c_index(time, event, pred[:, 0])
    hybrid = 0.3 * cidx + 0.7 * (1.0 - wb)
    return hybrid, {"c_index": cidx, "brier24": b24, "brier48": b48, "brier72": b72, "weighted_brier": wb}


def safe_mean(x, default=0.5):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return float(default)
    return float(np.clip(x.mean(), EPS, 1.0 - EPS))


def engineering(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    num_cols = [c for c in out.columns if c != "event_id" and pd.api.types.is_numeric_dtype(out[c])]
    for c in num_cols:
        out[f"{c}_na"] = out[c].isna().astype(int)

    if "dist_min_ci_0_5h" in out.columns:
        out["dist_km"] = out["dist_min_ci_0_5h"] / 1000.0
        out["log_dist"] = np.log1p(np.clip(out["dist_min_ci_0_5h"], 0, None))
    if "closing_speed_m_per_h" in out.columns:
        out["closing_speed_pos"] = np.clip(out["closing_speed_m_per_h"], 0, None)
        out["closing_speed_neg"] = np.clip(-out["closing_speed_m_per_h"], 0, None)
    if "along_track_speed" in out.columns:
        out["along_track_speed_pos"] = np.clip(out["along_track_speed"], 0, None)
        out["along_track_speed_neg"] = np.clip(-out["along_track_speed"], 0, None)
    if "radial_growth_rate_m_per_h" in out.columns:
        out["radial_growth_rate_pos"] = np.clip(out["radial_growth_rate_m_per_h"], 0, None)
    if "projected_advance_m" in out.columns and "dt_first_last_0_5h" in out.columns:
        out["projected_advance_rate"] = out["projected_advance_m"] / np.clip(out["dt_first_last_0_5h"], 0.25, None)
    if "dist_min_ci_0_5h" in out.columns and "closing_speed_pos" in out.columns:
        out["eta_close_h"] = out["dist_min_ci_0_5h"] / np.clip(out["closing_speed_pos"], 1.0, None)
        out["eta_close_h"] = out["eta_close_h"].clip(0, 500)
    if "dist_min_ci_0_5h" in out.columns and "along_track_speed_pos" in out.columns:
        out["eta_along_h"] = out["dist_min_ci_0_5h"] / np.clip(out["along_track_speed_pos"], 1.0, None)
        out["eta_along_h"] = out["eta_along_h"].clip(0, 500)
    if {"dist_min_ci_0_5h", "projected_advance_m"}.issubset(out.columns):
        out["remaining_after_projection_m"] = out["dist_min_ci_0_5h"] - out["projected_advance_m"]
        out["projection_ratio"] = out["projected_advance_m"] / np.clip(out["dist_min_ci_0_5h"], 1.0, None)
    if {"alignment_abs", "closing_speed_pos"}.issubset(out.columns):
        out["aligned_closing"] = out["alignment_abs"] * out["closing_speed_pos"]
    if {"alignment_abs", "projected_advance_m"}.issubset(out.columns):
        out["aligned_advance"] = out["alignment_abs"] * out["projected_advance_m"]
    if {"alignment_abs", "radial_growth_rate_pos"}.issubset(out.columns):
        out["aligned_radial_growth"] = out["alignment_abs"] * out["radial_growth_rate_pos"]
    if {"dist_km", "aligned_closing"}.issubset(out.columns):
        out["threat_score"] = out["aligned_closing"] / (out["dist_km"] + 0.25)
    if {"dist_km", "aligned_advance"}.issubset(out.columns):
        out["threat_score_2"] = out["aligned_advance"] / (out["dist_km"] + 0.25)
    if {"radial_growth_m", "dist_min_ci_0_5h"}.issubset(out.columns):
        out["growth_to_distance"] = out["radial_growth_m"] / np.clip(out["dist_min_ci_0_5h"], 1.0, None)
    if {"area_first_ha", "dist_km"}.issubset(out.columns):
        out["area_to_distance"] = out["area_first_ha"] / (out["dist_km"] + 0.25)
    if {"num_perimeters_0_5h", "dt_first_last_0_5h"}.issubset(out.columns):
        out["temporal_density"] = out["num_perimeters_0_5h"] / np.clip(out["dt_first_last_0_5h"], 0.25, None)
    if {"cross_track_component", "along_track_speed"}.issubset(out.columns):
        out["cross_vs_along"] = np.abs(out["cross_track_component"]) / (np.abs(out["along_track_speed"]) + 1.0)
    if "event_start_hour" in out.columns:
        out["hour_sin"] = np.sin(2 * np.pi * out["event_start_hour"] / 24.0)
        out["hour_cos"] = np.cos(2 * np.pi * out["event_start_hour"] / 24.0)
    if "event_start_dayofweek" in out.columns:
        out["dow_sin"] = np.sin(2 * np.pi * out["event_start_dayofweek"] / 7.0)
        out["dow_cos"] = np.cos(2 * np.pi * out["event_start_dayofweek"] / 7.0)
    if "event_start_month" in out.columns:
        out["month_sin"] = np.sin(2 * np.pi * out["event_start_month"] / 12.0)
        out["month_cos"] = np.cos(2 * np.pi * out["event_start_month"] / 12.0)

    if "dist_km" in out.columns:
        out["dist_band"] = pd.cut(
            out["dist_km"],
            bins=[-np.inf, 5, 10, 20, 50, np.inf],
            labels=["vnear", "near", "mid", "far", "vfar"]
        ).astype(str)
    else:
        out["dist_band"] = "unknown"

    if "eta_close_h" in out.columns:
        out["eta_band"] = pd.cut(
            out["eta_close_h"],
            bins=[-np.inf, 6, 12, 24, 48, 72, np.inf],
            labels=["eta_6", "eta_12", "eta_24", "eta_48", "eta_72", "eta_far"]
        ).astype(str)
    else:
        out["eta_band"] = "unknown"

    if "event_start_month" in out.columns:
        out["season_band"] = pd.cut(
            out["event_start_month"],
            bins=[0, 3, 6, 9, 12],
            labels=["q1", "q2", "q3", "q4"],
            include_lowest=True
        ).astype(str)
    else:
        out["season_band"] = "unknown"

    return out


def build_numeric_matrices(train_feat: pd.DataFrame, test_feat: pd.DataFrame):
    cat_cols = [c for c in train_feat.columns if c != "event_id" and train_feat[c].dtype == "object"]
    all_feat = pd.concat([train_feat.drop(columns=["event_id"]), test_feat.drop(columns=["event_id"])], axis=0, ignore_index=True)
    all_num = pd.get_dummies(all_feat, columns=cat_cols, dummy_na=True)
    x_train = all_num.iloc[: len(train_feat)].copy()
    x_test = all_num.iloc[len(train_feat):].copy()
    med = x_train.median(numeric_only=True)
    x_train = x_train.fillna(med)
    x_test = x_test.fillna(med)
    std = x_train.std(numeric_only=True).replace(0, 1.0).fillna(1.0)
    mean = x_train.mean(numeric_only=True)
    x_train_std = (x_train - mean) / std
    x_test_std = (x_test - mean) / std
    return x_train, x_test, x_train_std, x_test_std, cat_cols


def get_cat_matrices(train_feat: pd.DataFrame, test_feat: pd.DataFrame):
    x_train = train_feat.drop(columns=["event_id"]).copy()
    x_test = test_feat.drop(columns=["event_id"]).copy()
    cat_cols = [c for c in x_train.columns if x_train[c].dtype == "object"]
    for c in cat_cols:
        x_train[c] = x_train[c].fillna("NA").astype(str)
        x_test[c] = x_test[c].fillna("NA").astype(str)
    return x_train, x_test, cat_cols


def fit_predict_cat_binary(x_tr, y_tr, x_va, x_te, cat_cols, seed=2026, sample_weight=None, depth=4, iters=900):
    if not HAS_CATBOOST:
        raise RuntimeError("CatBoost is not available.")
    prior = safe_mean(y_tr)
    if len(np.unique(y_tr)) < 2 or len(y_tr) < 10:
        return np.full(len(x_va), prior), np.full(len(x_te), prior)
    model = CatBoostClassifier(
        iterations=iters,
        learning_rate=0.035,
        depth=depth,
        loss_function="Logloss",
        eval_metric="Logloss",
        l2_leaf_reg=8.0,
        random_strength=1.0,
        bootstrap_type="Bernoulli",
        subsample=0.85,
        rsm=0.85,
        random_seed=seed,
        allow_writing_files=False,
        verbose=False
    )
    tr_pool = Pool(x_tr, y_tr, cat_features=cat_cols, weight=sample_weight)
    va_pool = Pool(x_va, cat_features=cat_cols)
    te_pool = Pool(x_te, cat_features=cat_cols)
    model.fit(tr_pool)
    return model.predict_proba(va_pool)[:, 1], model.predict_proba(te_pool)[:, 1]


def fit_predict_cat_multiclass(x_tr, y_tr, x_va, x_te, cat_cols, seed=2026, classes_count=4, class_weights=None, depth=4, iters=900):
    if not HAS_CATBOOST:
        raise RuntimeError("CatBoost is not available.")
    if len(y_tr) < 8 or len(np.unique(y_tr)) < 2:
        prior = np.ones((len(x_va), classes_count)) / classes_count
        prior_te = np.ones((len(x_te), classes_count)) / classes_count
        return prior, prior_te
    params = dict(
        iterations=iters,
        learning_rate=0.035,
        depth=depth,
        loss_function="MultiClass",
        eval_metric="MultiClass",
        l2_leaf_reg=10.0,
        random_strength=1.0,
        bootstrap_type="Bernoulli",
        subsample=0.85,
        rsm=0.85,
        classes_count=classes_count,
        random_seed=seed,
        allow_writing_files=False,
        verbose=False
    )
    if class_weights is not None:
        params["class_weights"] = class_weights
    model = CatBoostClassifier(**params)
    tr_pool = Pool(x_tr, y_tr, cat_features=cat_cols)
    va_pool = Pool(x_va, cat_features=cat_cols)
    te_pool = Pool(x_te, cat_features=cat_cols)
    model.fit(tr_pool)
    p_va = np.asarray(model.predict_proba(va_pool), dtype=float)
    p_te = np.asarray(model.predict_proba(te_pool), dtype=float)
    if p_va.ndim == 1:
        p_va = np.column_stack([1 - p_va, p_va])
    if p_te.ndim == 1:
        p_te = np.column_stack([1 - p_te, p_te])
    # safety
    if p_va.shape[1] != classes_count:
        out_va = np.full((len(x_va), classes_count), 1.0 / classes_count)
        out_te = np.full((len(x_te), classes_count), 1.0 / classes_count)
        cols = min(classes_count, p_va.shape[1])
        out_va[:, :cols] = p_va[:, :cols]
        out_te[:, :cols] = p_te[:, :cols]
        p_va, p_te = out_va, out_te
    p_va = np.clip(p_va, EPS, None)
    p_va = p_va / p_va.sum(axis=1, keepdims=True)
    p_te = np.clip(p_te, EPS, None)
    p_te = p_te / p_te.sum(axis=1, keepdims=True)
    return p_va, p_te


def fit_predict_xgb_binary(x_tr, y_tr, x_va, x_te, seed=2026, sample_weight=None, depth=3, n_estimators=350):
    if not HAS_XGBOOST:
        raise RuntimeError("XGBoost is not available.")
    prior = safe_mean(y_tr)
    if len(np.unique(y_tr)) < 2 or len(y_tr) < 10:
        return np.full(len(x_va), prior), np.full(len(x_te), prior)
    pos = max(int(np.sum(y_tr == 1)), 1)
    neg = max(int(np.sum(y_tr == 0)), 1)
    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=depth,
        learning_rate=0.035,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=4.0,
        gamma=0.0,
        reg_alpha=0.15,
        reg_lambda=1.5,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=1.0,
        tree_method="hist",
        max_bin=256,
        random_state=seed,
        n_jobs=-1,
        verbosity=0
    )
    model.fit(x_tr, y_tr, sample_weight=sample_weight)
    return model.predict_proba(x_va)[:, 1], model.predict_proba(x_te)[:, 1]


def fit_predict_lr_binary(x_tr, y_tr, x_va, x_te, seed=2026, sample_weight=None):
    prior = safe_mean(y_tr)
    if len(np.unique(y_tr)) < 2 or len(y_tr) < 10:
        return np.full(len(x_va), prior), np.full(len(x_te), prior)
    model = LogisticRegression(
        C=0.7,
        solver="lbfgs",
        max_iter=5000,
        class_weight=None,
        random_state=seed
    )
    model.fit(x_tr, y_tr, sample_weight=sample_weight)
    return model.predict_proba(x_va)[:, 1], model.predict_proba(x_te)[:, 1]


def aft_cdf(pred_location: np.ndarray, horizons, distribution: str = "normal", scale: float = 1.0) -> np.ndarray:
    log_t = np.log(np.asarray(horizons, dtype=float))[None, :]
    z = (log_t - pred_location[:, None]) / scale
    if distribution == "normal":
        out = norm.cdf(z)
    elif distribution == "logistic":
        out = expit(z)
    elif distribution == "extreme":
        out = 1.0 - np.exp(-np.exp(z))
    else:
        raise ValueError(f"Unknown AFT distribution: {distribution}")
    return np.clip(out, EPS, 1.0 - EPS)


def fit_predict_xgb_aft(x_tr, time_tr, event_tr, x_va, x_te, distribution="normal", scale=1.0, seed=2026, rounds=500):
    if not HAS_XGBOOST:
        raise RuntimeError("XGBoost is not available.")
    lower = np.clip(np.asarray(time_tr, dtype=float), 1e-3, None)
    upper = np.where(np.asarray(event_tr) == 1, lower, np.inf)
    dtr = xgb.DMatrix(x_tr)
    dtr.set_float_info("label_lower_bound", lower)
    dtr.set_float_info("label_upper_bound", upper)
    dva = xgb.DMatrix(x_va)
    dte = xgb.DMatrix(x_te)
    params = {
        "objective": "survival:aft",
        "eval_metric": "aft-nloglik",
        "aft_loss_distribution": distribution,
        "aft_loss_distribution_scale": float(scale),
        "learning_rate": 0.035,
        "max_depth": 3,
        "min_child_weight": 4.0,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "lambda": 1.5,
        "alpha": 0.15,
        "tree_method": "hist",
        "max_bin": 256,
        "verbosity": 0,
        "seed": seed,
    }
    booster = xgb.train(params, dtr, num_boost_round=rounds, verbose_eval=False)
    p_va = aft_cdf(booster.predict(dva), HORIZONS, distribution=distribution, scale=scale)
    p_te = aft_cdf(booster.predict(dte), HORIZONS, distribution=distribution, scale=scale)
    return p_va, p_te


def build_long_train(x_raw: pd.DataFrame, time: np.ndarray, event: np.ndarray):
    rows = []
    labels = []
    weights = []
    cuts = [0, 12, 24, 48, 72]
    interval_weights = [1.35, 1.15, 1.0, 0.9]
    for i in range(len(x_raw)):
        t = float(time[i])
        e = int(event[i])
        base = x_raw.iloc[i].to_dict()
        for j in range(4):
            start, end = cuts[j], cuts[j + 1]
            if t < start:
                break
            if e == 0 and t < end:
                break
            row = dict(base)
            row["interval_id"] = f"I{j}"
            row["interval_start"] = start
            row["interval_end"] = end
            row["interval_len"] = end - start
            row["interval_mid"] = 0.5 * (start + end)
            if "eta_close_h" in row:
                row["eta_minus_interval_end"] = float(row["eta_close_h"]) - end if pd.notna(row["eta_close_h"]) else np.nan
            rows.append(row)
            if e == 1 and t <= end:
                labels.append(1)
                weights.append(interval_weights[j] * 1.5)
                break
            else:
                labels.append(0)
                weights.append(interval_weights[j])
    x_long = pd.DataFrame(rows)
    y_long = np.asarray(labels, dtype=int)
    w_long = np.asarray(weights, dtype=float)
    return x_long, y_long, w_long


def build_long_pred(x_raw: pd.DataFrame):
    rows = []
    sample_ids = []
    for i in range(len(x_raw)):
        base = x_raw.iloc[i].to_dict()
        for j, (start, end) in enumerate(zip([0, 12, 24, 48], [12, 24, 48, 72])):
            row = dict(base)
            row["interval_id"] = f"I{j}"
            row["interval_start"] = start
            row["interval_end"] = end
            row["interval_len"] = end - start
            row["interval_mid"] = 0.5 * (start + end)
            if "eta_close_h" in row:
                row["eta_minus_interval_end"] = float(row["eta_close_h"]) - end if pd.notna(row["eta_close_h"]) else np.nan
            rows.append(row)
            sample_ids.append(i)
    x_long = pd.DataFrame(rows)
    sample_ids = np.asarray(sample_ids, dtype=int)
    return x_long, sample_ids


def hazard_to_cumulative(hazard_pred: np.ndarray, n_samples: int) -> np.ndarray:
    hazard_pred = np.asarray(hazard_pred, dtype=float).reshape(n_samples, 4)
    hazard_pred = np.clip(hazard_pred, EPS, 1.0 - EPS)
    cum = 1.0 - np.cumprod(1.0 - hazard_pred, axis=1)
    return monotone_clip(cum)


def run_direct_family(x_cat_tr, x_cat_te, cat_cols, x_num_tr, x_num_te, x_std_tr, x_std_te, time, event, splits):
    n = len(x_cat_tr)
    m = len(x_cat_te)
    oof = np.zeros((n, 4), dtype=float)
    cnt = np.zeros((n, 4), dtype=float)
    pred_te = np.zeros((m, 4), dtype=float)

    knowns = {}
    labels = {}
    for h in HORIZONS:
        knowns[h], labels[h] = known_and_label(time, event, h)

    for split_id, (tr_idx, va_idx) in enumerate(splits):
        print(f"[Direct] split {split_id + 1}/{len(splits)}")
        for h_idx, h in enumerate(HORIZONS):
            known = knowns[h]
            y = labels[h]
            tr_known = tr_idx[known[tr_idx]]
            y_tr = y[tr_known]
            pos = max(int((y_tr == 1).sum()), 1)
            neg = max(int((y_tr == 0).sum()), 1)
            pos_w = float(np.clip(np.sqrt(neg / pos), 1.0, 4.0))
            sw = np.where(y_tr == 1, pos_w, 1.0)

            val_preds = []
            test_preds = []

            if HAS_CATBOOST:
                try:
                    pv, pt = fit_predict_cat_binary(
                        x_cat_tr.iloc[tr_known], y_tr,
                        x_cat_tr.iloc[va_idx], x_cat_te,
                        cat_cols=cat_cols,
                        seed=SEED + split_id * 17 + h,
                        sample_weight=sw,
                        depth=4 if h <= 24 else 5,
                        iters=900 if h <= 24 else 1000,
                    )
                    val_preds.append(pv)
                    test_preds.append(pt)
                except Exception as e:
                    print(f"  CatBoost failed at horizon={h}: {e}")

            if HAS_XGBOOST:
                try:
                    pv, pt = fit_predict_xgb_binary(
                        x_num_tr.iloc[tr_known], y_tr,
                        x_num_tr.iloc[va_idx], x_num_te,
                        seed=SEED + split_id * 19 + h,
                        sample_weight=sw,
                        depth=3 if h <= 24 else 4,
                        n_estimators=350 if h <= 24 else 425,
                    )
                    val_preds.append(pv)
                    test_preds.append(pt)
                except Exception as e:
                    print(f"  XGBoost failed at horizon={h}: {e}")

            try:
                pv, pt = fit_predict_lr_binary(
                    x_std_tr.iloc[tr_known], y_tr,
                    x_std_tr.iloc[va_idx], x_std_te,
                    seed=SEED + split_id * 23 + h,
                    sample_weight=sw,
                )
                val_preds.append(pv)
                test_preds.append(pt)
            except Exception as e:
                print(f"  Logistic failed at horizon={h}: {e}")

            if len(val_preds) == 0:
                prior = safe_mean(y_tr)
                pv = np.full(len(va_idx), prior)
                pt = np.full(m, prior)
            else:
                pv = np.mean(np.column_stack(val_preds), axis=1)
                pt = np.mean(np.column_stack(test_preds), axis=1)

            oof[va_idx, h_idx] += pv
            cnt[va_idx, h_idx] += 1.0
            pred_te[:, h_idx] += pt

    oof = np.divide(oof, np.maximum(cnt, 1.0))
    pred_te /= len(splits)
    return monotone_clip(oof), monotone_clip(pred_te)


def run_hazard_family(x_cat_tr, x_cat_te, time, event, splits):
    n = len(x_cat_tr)
    m = len(x_cat_te)
    oof = np.zeros((n, 4), dtype=float)
    cnt = np.zeros((n, 4), dtype=float)
    pred_te = np.zeros((m, 4), dtype=float)

    for split_id, (tr_idx, va_idx) in enumerate(splits):
        print(f"[Hazard] split {split_id + 1}/{len(splits)}")
        x_long_tr, y_long_tr, w_long_tr = build_long_train(x_cat_tr.iloc[tr_idx].reset_index(drop=True), time[tr_idx], event[tr_idx])
        x_long_va, va_ids = build_long_pred(x_cat_tr.iloc[va_idx].reset_index(drop=True))
        x_long_te, te_ids = build_long_pred(x_cat_te.reset_index(drop=True))

        cat_cols_long = [c for c in x_long_tr.columns if x_long_tr[c].dtype == "object"]
        for c in cat_cols_long:
            x_long_tr[c] = x_long_tr[c].fillna("NA").astype(str)
            x_long_va[c] = x_long_va[c].fillna("NA").astype(str)
            x_long_te[c] = x_long_te[c].fillna("NA").astype(str)

        preds_val = []
        preds_te = []

        if HAS_CATBOOST:
            try:
                pv, pt = fit_predict_cat_binary(
                    x_long_tr, y_long_tr,
                    x_long_va, x_long_te,
                    cat_cols=cat_cols_long,
                    seed=SEED + split_id * 29,
                    sample_weight=w_long_tr,
                    depth=5,
                    iters=1050,
                )
                preds_val.append(pv)
                preds_te.append(pt)
            except Exception as e:
                print(f"  Hazard CatBoost failed: {e}")

        if len(preds_val) == 0:
            prior = safe_mean(y_long_tr)
            pv = np.full(len(x_long_va), prior)
            pt = np.full(len(x_long_te), prior)
        else:
            pv = np.mean(np.column_stack(preds_val), axis=1)
            pt = np.mean(np.column_stack(preds_te), axis=1)

        cum_va = hazard_to_cumulative(pv, len(va_idx))
        cum_te = hazard_to_cumulative(pt, m)

        oof[va_idx] += cum_va
        cnt[va_idx] += 1.0
        pred_te += cum_te

    oof = np.divide(oof, np.maximum(cnt, 1.0))
    pred_te /= len(splits)
    return monotone_clip(oof), monotone_clip(pred_te)


def run_aft_family(x_num_tr, x_num_te, time, event, splits):
    n = len(x_num_tr)
    m = len(x_num_te)
    oof = np.zeros((n, 4), dtype=float)
    cnt = np.zeros((n, 4), dtype=float)
    pred_te = np.zeros((m, 4), dtype=float)

    aft_specs = [
        ("normal", 1.0, 450),
        ("logistic", 1.2, 450),
        ("extreme", 0.9, 500),
    ]

    for split_id, (tr_idx, va_idx) in enumerate(splits):
        print(f"[AFT] split {split_id + 1}/{len(splits)}")
        fold_val_preds = []
        fold_test_preds = []
        for dist_name, scale, rounds in aft_specs:
            if not HAS_XGBOOST:
                continue
            try:
                pv, pt = fit_predict_xgb_aft(
                    x_num_tr.iloc[tr_idx], time[tr_idx], event[tr_idx],
                    x_num_tr.iloc[va_idx], x_num_te,
                    distribution=dist_name,
                    scale=scale,
                    seed=SEED + split_id * 31 + int(scale * 10),
                    rounds=rounds,
                )
                fold_val_preds.append(pv)
                fold_test_preds.append(pt)
            except Exception as e:
                print(f"  AFT failed ({dist_name}, scale={scale}): {e}")

        if len(fold_val_preds) == 0:
            # fallback to a very simple heuristic using 72h hit rate
            rate = safe_mean((event[tr_idx] == 1).astype(float))
            pv = np.full((len(va_idx), 4), [0.25 * rate, 0.55 * rate, 0.85 * rate, rate], dtype=float)
            pt = np.full((m, 4), [0.25 * rate, 0.55 * rate, 0.85 * rate, rate], dtype=float)
        else:
            pv = np.mean(np.stack(fold_val_preds, axis=0), axis=0)
            pt = np.mean(np.stack(fold_test_preds, axis=0), axis=0)

        oof[va_idx] += pv
        cnt[va_idx] += 1.0
        pred_te += pt

    oof = np.divide(oof, np.maximum(cnt, 1.0))
    pred_te /= len(splits)
    return monotone_clip(oof), monotone_clip(pred_te)


def event_bin(time_values: np.ndarray) -> np.ndarray:
    bins = np.zeros(len(time_values), dtype=int)
    bins[(time_values > 12) & (time_values <= 24)] = 1
    bins[(time_values > 24) & (time_values <= 48)] = 2
    bins[(time_values > 48)] = 3
    return bins


def run_timing_family(x_cat_tr, x_cat_te, time, event, splits):
    n = len(x_cat_tr)
    m = len(x_cat_te)
    oof = np.zeros((n, 4), dtype=float)
    cnt = np.zeros((n, 4), dtype=float)
    pred_te = np.zeros((m, 4), dtype=float)

    known72, label72 = known_and_label(time, event, 72)

    for split_id, (tr_idx, va_idx) in enumerate(splits):
        print(f"[Timing] split {split_id + 1}/{len(splits)}")
        tr_known = tr_idx[known72[tr_idx]]
        y72 = label72[tr_known]

        p72_val_list = []
        p72_te_list = []

        base_cat_cols = [c for c in x_cat_tr.columns if x_cat_tr[c].dtype == "object"]

        if HAS_CATBOOST:
            try:
                pv72, pt72 = fit_predict_cat_binary(
                    x_cat_tr.iloc[tr_known], y72,
                    x_cat_tr.iloc[va_idx], x_cat_te,
                    cat_cols=base_cat_cols,
                    seed=SEED + split_id * 37,
                    sample_weight=np.where(y72 == 1, np.clip(np.sqrt((len(y72) - y72.sum() + 1) / (y72.sum() + 1)), 1.0, 4.0), 1.0),
                    depth=4,
                    iters=950,
                )
                p72_val_list.append(pv72)
                p72_te_list.append(pt72)
            except Exception as e:
                print(f"  Timing 72h CatBoost failed: {e}")

        if len(p72_val_list) == 0:
            p72_val = np.full(len(va_idx), safe_mean(y72))
            p72_te = np.full(m, safe_mean(y72))
        else:
            p72_val = np.mean(np.column_stack(p72_val_list), axis=1)
            p72_te = np.mean(np.column_stack(p72_te_list), axis=1)

        hit_tr = tr_idx[event[tr_idx] == 1]
        ybin_tr = event_bin(time[hit_tr])
        counts = np.bincount(ybin_tr, minlength=4)
        total = counts.sum()
        class_weights = []
        for c in counts:
            if c <= 0:
                class_weights.append(1.0)
            else:
                class_weights.append(float(np.clip(np.sqrt(total / c), 1.0, 4.0)))

        if HAS_CATBOOST and len(hit_tr) >= 8 and np.unique(ybin_tr).size >= 2:
            try:
                pbin_val, pbin_te = fit_predict_cat_multiclass(
                    x_cat_tr.iloc[hit_tr], ybin_tr,
                    x_cat_tr.iloc[va_idx], x_cat_te,
                    cat_cols=base_cat_cols,
                    seed=SEED + split_id * 41,
                    classes_count=4,
                    class_weights=class_weights,
                    depth=4,
                    iters=900,
                )
            except Exception as e:
                print(f"  Timing multiclass CatBoost failed: {e}")
                prior_bins = np.bincount(ybin_tr, minlength=4).astype(float)
                prior_bins = prior_bins / np.clip(prior_bins.sum(), 1.0, None)
                pbin_val = np.tile(prior_bins, (len(va_idx), 1))
                pbin_te = np.tile(prior_bins, (m, 1))
        else:
            prior_bins = np.bincount(ybin_tr, minlength=4).astype(float)
            prior_bins = prior_bins / np.clip(prior_bins.sum(), 1.0, None)
            pbin_val = np.tile(prior_bins, (len(va_idx), 1))
            pbin_te = np.tile(prior_bins, (m, 1))

        cum_val = np.cumsum(pbin_val, axis=1) * p72_val[:, None]
        cum_te = np.cumsum(pbin_te, axis=1) * p72_te[:, None]

        oof[va_idx] += monotone_clip(cum_val)
        cnt[va_idx] += 1.0
        pred_te += monotone_clip(cum_te)

    oof = np.divide(oof, np.maximum(cnt, 1.0))
    pred_te /= len(splits)
    return monotone_clip(oof), monotone_clip(pred_te)


def random_search_blend(base_oof: dict, time: np.ndarray, event: np.ndarray, n_search: int = 5000, seed: int = 2026):
    names = list(base_oof.keys())
    preds = np.stack([base_oof[n] for n in names], axis=0)  # [m, n, h]
    n_models = preds.shape[0]
    rng = np.random.default_rng(seed)

    def blend_with_w(wmat):
        out = np.zeros_like(next(iter(base_oof.values())))
        for h in range(4):
            out[:, h] = np.sum(preds[:, :, h].T * wmat[h], axis=1)
        return monotone_clip(out)

    # individual model scores for biasing the search
    model_scores = np.array([hybrid_score(time, event, base_oof[n])[0] for n in names], dtype=float)
    alpha_base = 1.0 + 12.0 * np.clip(model_scores - model_scores.min() + 1e-4, 1e-4, None)
    alpha_base = alpha_base / alpha_base.mean()

    uniform_w = np.full((4, n_models), 1.0 / n_models)
    best_w = uniform_w.copy()
    best_pred = blend_with_w(best_w)
    best_score, _ = hybrid_score(time, event, best_pred)

    for _ in range(n_search):
        wmat = np.zeros((4, n_models), dtype=float)
        for h in range(4):
            alpha = 0.55 * alpha_base + 0.45 * np.ones_like(alpha_base)
            draw = rng.dirichlet(alpha)
            wmat[h] = draw / draw.sum()
        pred = blend_with_w(wmat)
        score, _ = hybrid_score(time, event, pred)
        if score > best_score:
            best_score = score
            best_w = wmat.copy()
            best_pred = pred.copy()

    return names, best_w, best_pred, best_score


def subgroup_family_prior(base_oof: dict, time: np.ndarray, event: np.ndarray, mask: np.ndarray):
    vals = []
    names = list(base_oof.keys())
    for name in names:
        score, _ = hybrid_score(time[mask], event[mask], base_oof[name][mask])
        vals.append(score)
    vals = np.asarray(vals, dtype=float)
    vals = np.exp((vals - vals.max()) / 0.02)
    vals = vals / np.clip(vals.sum(), EPS, None)
    return names, vals


def apply_weight_matrix(base_pred: dict, names: list, wmat: np.ndarray):
    out = np.zeros_like(next(iter(base_pred.values())))
    for h in range(4):
        tmp = np.zeros(out.shape[0], dtype=float)
        for j, name in enumerate(names):
            tmp += wmat[h, j] * base_pred[name][:, h]
        out[:, h] = tmp
    return monotone_clip(out)


def tune_distance_stratified(base_oof: dict, base_te: dict, global_names: list, global_w: np.ndarray, train_feat: pd.DataFrame, time: np.ndarray, event: np.ndarray):
    dist_series = train_feat["dist_km"].replace([np.inf, -np.inf], np.nan).fillna(train_feat["dist_km"].median())
    quants = np.unique(np.quantile(dist_series, np.linspace(0.2, 0.8, N_NEARFAR_GRID))).tolist()

    global_oof = apply_weight_matrix(base_oof, global_names, global_w)
    global_te = apply_weight_matrix(base_te, global_names, global_w)
    best_pred = global_oof.copy()
    best_te = global_te.copy()
    best_score, _ = hybrid_score(time, event, best_pred)
    best_info = {"mode": "global", "threshold": None, "gamma_near": 0.0, "gamma_far": 0.0}

    for thr in quants:
        near_mask = dist_series.values <= thr
        far_mask = ~near_mask
        if near_mask.sum() < 30 or far_mask.sum() < 30:
            continue

        names_near, prior_near = subgroup_family_prior(base_oof, time, event, near_mask)
        names_far, prior_far = subgroup_family_prior(base_oof, time, event, far_mask)
        assert names_near == global_names == names_far

        prior_near_mat = np.vstack([prior_near] * 4)
        prior_far_mat = np.vstack([prior_far] * 4)

        for gamma_near in np.linspace(0.0, 0.75, 6):
            for gamma_far in np.linspace(0.0, 0.75, 6):
                w_near = (1.0 - gamma_near) * global_w + gamma_near * prior_near_mat
                w_far = (1.0 - gamma_far) * global_w + gamma_far * prior_far_mat
                w_near = w_near / np.clip(w_near.sum(axis=1, keepdims=True), EPS, None)
                w_far = w_far / np.clip(w_far.sum(axis=1, keepdims=True), EPS, None)

                pred_oof = np.zeros_like(global_oof)
                pred_te = np.zeros_like(global_te)
                pred_oof[near_mask] = apply_weight_matrix({k: v[near_mask] for k, v in base_oof.items()}, global_names, w_near)
                pred_oof[far_mask] = apply_weight_matrix({k: v[far_mask] for k, v in base_oof.items()}, global_names, w_far)
                pred_te_mask_near = (base_te["direct"][:, 0] * 0 + 1).astype(bool)  # all test rows
                # decide near/far on test using train threshold later outside
                score, _ = hybrid_score(time, event, pred_oof)
                if score > best_score:
                    best_score = score
                    best_pred = pred_oof.copy()
                    best_info = {"mode": "distance_stratified", "threshold": float(thr), "gamma_near": float(gamma_near), "gamma_far": float(gamma_far), "w_near": w_near.copy(), "w_far": w_far.copy()}

    # prepare test predictions
    if best_info["mode"] == "global":
        return best_pred, best_te, best_info, best_score

    # infer test-side near/far using same threshold on test dist_km
    return best_pred, None, best_info, best_score


def crossfit_platt(raw_oof: np.ndarray, raw_te: np.ndarray, time: np.ndarray, event: np.ndarray, horizon: int, seed: int = 2026):
    known, label = known_and_label(time, event, horizon)
    idx = np.where(known)[0]
    if len(idx) < 30 or np.unique(label[idx]).size < 2:
        return raw_oof.copy(), raw_te.copy(), "raw"

    x = np.clip(raw_oof[idx], 1e-6, 1 - 1e-6)
    z = np.clip(logit(x), -12, 12).reshape(-1, 1)
    y = label[idx]

    counts = np.bincount(y.astype(int), minlength=2)
    if counts.min() < 2:
        return raw_oof.copy(), raw_te.copy(), "raw"
    skf = StratifiedKFold(n_splits=min(5, int(counts.min())), shuffle=True, random_state=seed)
    oof_cal = raw_oof.copy()
    pred_cf = np.zeros(len(idx), dtype=float)
    for tr, va in skf.split(z, y):
        lr = LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000, class_weight="balanced", random_state=seed)
        lr.fit(z[tr], y[tr])
        pred_cf[va] = lr.predict_proba(z[va])[:, 1]
    oof_cal[idx] = pred_cf

    # fit full calibrator for test
    lr_full = LogisticRegression(C=1.0, solver="lbfgs", max_iter=5000, class_weight="balanced", random_state=seed)
    lr_full.fit(z, y)
    z_te = np.clip(logit(np.clip(raw_te, 1e-6, 1 - 1e-6)), -12, 12).reshape(-1, 1)
    te_cal = lr_full.predict_proba(z_te)[:, 1]

    raw_brier = np.mean((raw_oof[idx] - y) ** 2)
    cal_brier = np.mean((oof_cal[idx] - y) ** 2)
    if cal_brier <= raw_brier:
        return oof_cal, te_cal, "platt"
    return raw_oof.copy(), raw_te.copy(), "raw"


def finalize_predictions(base_oof: dict, base_te: dict, train_feat: pd.DataFrame, test_feat: pd.DataFrame, time: np.ndarray, event: np.ndarray):
    names, global_w, global_oof, global_score = random_search_blend(base_oof, time, event, n_search=N_SEARCH, seed=SEED)
    global_te = apply_weight_matrix(base_te, names, global_w)
    print("\nGlobal blend score:", round(global_score, 6))
    for h, row in zip(HORIZONS, global_w):
        print(f"  Weights @ {h}h:", {name: round(float(w), 4) for name, w in zip(names, row)})

    strat_oof, _, strat_info, strat_score = tune_distance_stratified(base_oof, base_te, names, global_w, train_feat, time, event)
    if strat_score > global_score and strat_info["mode"] == "distance_stratified":
        print("\nDistance-stratified blend selected.")
        print(strat_info)
        thr = strat_info["threshold"]
        w_near = strat_info["w_near"]
        w_far = strat_info["w_far"]

        test_dist = test_feat["dist_km"].replace([np.inf, -np.inf], np.nan).fillna(train_feat["dist_km"].median()).values
        near_te_mask = test_dist <= thr

        strat_te = np.zeros_like(global_te)
        if near_te_mask.any():
            strat_te[near_te_mask] = apply_weight_matrix({k: v[near_te_mask] for k, v in base_te.items()}, names, w_near)
        if (~near_te_mask).any():
            strat_te[~near_te_mask] = apply_weight_matrix({k: v[~near_te_mask] for k, v in base_te.items()}, names, w_far)

        blend_oof = strat_oof
        blend_te = strat_te
        blend_info = strat_info
    else:
        print("\nGlobal blend retained.")
        blend_oof = global_oof
        blend_te = global_te
        blend_info = {"mode": "global", "w": global_w}

    # Calibrate 24/48/72 only. Keep 12h mostly raw to preserve ranking power.
    cal_oof = blend_oof.copy()
    cal_te = blend_te.copy()
    cal_info = {}
    for h_idx, h in enumerate(HORIZONS[1:], start=1):
        oof_h, te_h, method = crossfit_platt(blend_oof[:, h_idx], blend_te[:, h_idx], time, event, h, seed=SEED + h)
        cal_oof[:, h_idx] = oof_h
        cal_te[:, h_idx] = te_h
        cal_info[h] = method

    # 12h: ranking-preserving hedge, lightly tie it to the calibrated 24h ceiling
    cal_oof[:, 0] = np.minimum(cal_oof[:, 1], blend_oof[:, 0])
    cal_te[:, 0] = np.minimum(cal_te[:, 1], blend_te[:, 0])

    # Light shrinkage toward empirical rates; tuned on OOF per horizon
    for h_idx, h in enumerate(HORIZONS):
        known, label = known_and_label(time, event, h)
        prior = safe_mean(label[known], default=safe_mean(label))
        if h == 12:
            lam_grid = [1.0]
        else:
            lam_grid = np.linspace(0.88, 1.0, 7)

        best_lam = 1.0
        best_metric = np.inf
        best_oof_h = cal_oof[:, h_idx].copy()
        best_te_h = cal_te[:, h_idx].copy()

        for lam in lam_grid:
            cand_oof = lam * cal_oof[:, h_idx] + (1 - lam) * prior
            cand_te = lam * cal_te[:, h_idx] + (1 - lam) * prior
            if h == 12:
                metric = -harrell_c_index(time, event, np.minimum(cal_oof[:, 1], cand_oof))
            else:
                metric = np.mean((cand_oof[known] - label[known]) ** 2)
            if metric < best_metric:
                best_metric = metric
                best_lam = float(lam)
                best_oof_h = cand_oof
                best_te_h = cand_te

        cal_oof[:, h_idx] = best_oof_h
        cal_te[:, h_idx] = best_te_h
        cal_info[f"shrink_{h}"] = best_lam

    cal_oof = monotone_clip(cal_oof)
    cal_te = monotone_clip(cal_te)

    raw_score, raw_detail = hybrid_score(time, event, blend_oof)
    cal_score, cal_detail = hybrid_score(time, event, cal_oof)
    if cal_score >= raw_score:
        print("\nCalibrated blend kept.")
        print("Calibration methods:", cal_info)
        print("Raw:", raw_detail)
        print("Calibrated:", cal_detail)
        return cal_oof, cal_te, {"blend": blend_info, "calibration": cal_info, "score": cal_detail}
    else:
        print("\nRaw blend kept after calibration check.")
        print("Raw:", raw_detail)
        print("Calibrated:", cal_detail)
        return blend_oof, blend_te, {"blend": blend_info, "calibration": {"status": "skipped"}, "score": raw_detail}


def main():
    root, train, test, sample, meta = read_competition_files()

    target_cols = ["time_to_hit_hours", "event"]
    feature_cols = [c for c in train.columns if c not in ["event_id"] + target_cols]

    train_feat = engineering(train[["event_id"] + feature_cols].copy())
    test_feat = engineering(test[["event_id"] + feature_cols].copy())

    x_num_tr, x_num_te, x_std_tr, x_std_te, num_cat_cols = build_numeric_matrices(train_feat, test_feat)
    x_cat_tr, x_cat_te, cat_cols = get_cat_matrices(train_feat, test_feat)

    time = train["time_to_hit_hours"].astype(float).values
    event = train["event"].astype(int).values
    strata = make_strata(event, time)

    splitter = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
    splits = list(splitter.split(np.zeros(len(train)), strata))
    print(f"Generated {len(splits)} CV splits.")

    base_oof = {}
    base_te = {}

    # 1) Direct horizon family
    oof_direct, te_direct = run_direct_family(
        x_cat_tr, x_cat_te, cat_cols,
        x_num_tr, x_num_te,
        x_std_tr, x_std_te,
        time, event, splits
    )
    base_oof["direct"] = oof_direct
    base_te["direct"] = te_direct
    score, detail = hybrid_score(time, event, oof_direct)
    print("\nDirect family:", round(score, 6), detail)

    # 2) Discrete hazard family
    oof_hazard, te_hazard = run_hazard_family(x_cat_tr, x_cat_te, time, event, splits)
    base_oof["hazard"] = oof_hazard
    base_te["hazard"] = te_hazard
    score, detail = hybrid_score(time, event, oof_hazard)
    print("\nHazard family:", round(score, 6), detail)

    # 3) AFT survival family
    oof_aft, te_aft = run_aft_family(x_num_tr, x_num_te, time, event, splits)
    base_oof["aft"] = oof_aft
    base_te["aft"] = te_aft
    score, detail = hybrid_score(time, event, oof_aft)
    print("\nAFT family:", round(score, 6), detail)

    # 4) 72h hit + timing-bin family
    oof_timing, te_timing = run_timing_family(x_cat_tr, x_cat_te, time, event, splits)
    base_oof["timing"] = oof_timing
    base_te["timing"] = te_timing
    score, detail = hybrid_score(time, event, oof_timing)
    print("\nTiming family:", round(score, 6), detail)

    # Final blend + calibration
    final_oof, final_te, info = finalize_predictions(base_oof, base_te, train_feat, test_feat, time, event)
    final_score, final_detail = hybrid_score(time, event, final_oof)
    print("\nFinal OOF:", round(final_score, 6), final_detail)
    print(json.dumps(info, indent=2, default=lambda x: x.tolist() if isinstance(x, np.ndarray) else str(x)))

    # Submission
    pred_df = pd.DataFrame({
        "event_id": test["event_id"].astype(str).values,
        "prob_12h": final_te[:, 0],
        "prob_24h": final_te[:, 1],
        "prob_48h": final_te[:, 2],
        "prob_72h": final_te[:, 3],
    })
    pred_df = pred_df.groupby("event_id", as_index=False).mean()

    sub = sample[["event_id"]].copy()
    sub["event_id"] = sub["event_id"].astype(str)
    sub = sub.merge(pred_df, on="event_id", how="left")

    for c in ["prob_12h", "prob_24h", "prob_48h", "prob_72h"]:
        if sub[c].isna().any():
            fill_val = float(pred_df[c].mean())
            sub[c] = sub[c].fillna(fill_val)

    probs = sub[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].values
    probs = monotone_clip(probs)
    sub[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]] = probs

    # Strict validation
    assert list(sub.columns) == ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]
    assert sub["event_id"].is_unique
    assert set(sub["event_id"]) == set(test["event_id"].astype(str))
    assert len(sub) == len(test)
    arr = sub[["prob_12h", "prob_24h", "prob_48h", "prob_72h"]].values
    assert np.all((arr >= 0) & (arr <= 1))
    assert np.all(arr[:, 0] <= arr[:, 1] + 1e-12)
    assert np.all(arr[:, 1] <= arr[:, 2] + 1e-12)
    assert np.all(arr[:, 2] <= arr[:, 3] + 1e-12)

    sub.to_csv("submission.csv", index=False)
    print("\nSaved submission.csv")
    print(sub.head())

    # Light artifact trail for debugging
    pd.DataFrame(final_oof, columns=["oof_12h", "oof_24h", "oof_48h", "oof_72h"]).to_csv("oof_predictions_debug.csv", index=False)

    del train_feat, test_feat, x_num_tr, x_num_te, x_std_tr, x_std_te, x_cat_tr, x_cat_te
    gc.collect()


if __name__ == "__main__":
    main()


Dataset root: /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
Train shape: (221, 37) | Test shape: (95, 35)
Generated 20 CV splits.
[Direct] split 1/20
[Direct] split 2/20
[Direct] split 3/20
[Direct] split 4/20
[Direct] split 5/20
[Direct] split 6/20
[Direct] split 7/20
[Direct] split 8/20
[Direct] split 9/20
[Direct] split 10/20
[Direct] split 11/20
[Direct] split 12/20
[Direct] split 13/20
[Direct] split 14/20
[Direct] split 15/20
[Direct] split 16/20
[Direct] split 17/20
[Direct] split 18/20
[Direct] split 19/20
[Direct] split 20/20

Direct family: 0.971507 {'c_index': 0.940377608479629, 'brier24': 0.029107155472931856, 'brier48': 0.016048379776525665, 'brier72': 9.999999989472883e-15, 'weighted_brier': 0.015151498552492822}
[Hazard] split 1/20
[Hazard] split 2/20
[Hazard] split 3/20
[Hazard] split 4/20
[Hazard] split 5/20
[Hazard] split 6/20
[Hazard] split 7/20
[Hazard] split 8/20
[Hazard] split 9/20
[Hazard] split 10/20
[Hazard] split 11/20
[Hazard] split 12/20
[Hazard] s